# 03 · Augmentation & generation
Build the augmented prompt from retrieved chunks (numbered context), generate with the **heavy** model, then verify grounding. This is the manual version of what the LangGraph agent automates in notebook 04.

In [1]:
# ---- Shared setup (identical in every notebook) -------------------------------
import os, sys, json, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "sample_data"))

# OFFLINE=True  -> deterministic hash embeddings + mock LLM + local Chroma. No keys, no network.
# OFFLINE=False -> REAL Gemini/Groq/OpenAI + Chroma Cloud from your .env  (set NB_OFFLINE=0 or edit below)
OFFLINE = os.getenv("NB_OFFLINE", "1") == "1"
if OFFLINE:
    os.environ["RAG_OFFLINE"] = "1"
    os.environ["CHROMA_LOCAL_DIR"] = str(ROOT / "chroma_notebook")   # shared by all notebooks
else:
    os.environ["RAG_OFFLINE"] = "0"

from app.config import get_settings
S = get_settings()
print("mode            :", "OFFLINE (mock)" if S.offline else "LIVE providers")
print("vector store    :", "Chroma Cloud" if S.use_chroma_cloud else f"local ({S.chroma_local_dir})")
print("embedding       :", "hash512" if S.offline else S.embedding_provider)
print("LLM providers   :", [p for p, k in (("gemini", S.gemini_key), ("groq", S.groq_key), ("openai", S.openai_key)) if k] or "none (offline mock)")

mode            : OFFLINE (mock)
vector store    : local (/home/claude/proj/v1/rag_service/chroma_notebook)
embedding       : hash512
LLM providers   : ['gemini', 'groq', 'openai']


In [2]:
from app.retriever import HybridRetriever, build_where
from app.agent import _ctx_block, NO_ANSWER
from app.providers import get_llm, llm_text
question = "Which organelle produces ATP and what kind of DNA does it have?"
hits = HybridRetriever(where=build_where(["BIO101"]), top_k=3).search(question)
relevant = [{"doc": d, "score": s} for d, s in hits]
print(_ctx_block(relevant)[:600], "...")

[1] (bio_notes.pdf, p.1)
Cell Biology Notes
Photosynthesis occurs in the chloroplasts of plant cells. Chlorophyll absorbs light energy, which is
used to convert carbon dioxide and water into glucose and oxygen.
Mitochondria are the organelles that produce most of the cell's ATP through cellular respiration.
They have a double membrane and their own circular DNA.
The cell membrane is a phospholipid bilayer. It is selectively permeable and controls which
substances enter and leave the cell. Osmosis is the diffusion of water across this membrane.
[2] (bio_slides.pptx, p.1)
Cell Biology Notes
Phot ...


## Augmented prompt

In [3]:
prompt = ("TASK:ANSWER\nYou are a university course tutor. Answer ONLY from the numbered context. Cite like [1]. "
          "If the context lacks the answer reply exactly INSUFFICIENT_CONTEXT.\n"
          f"CONTEXT:\n{_ctx_block(relevant)}\nQUESTION: {question}\n")
print(len(prompt), "chars sent to the LLM")

1749 chars sent to the LLM


## Generation (heavy tier)

In [4]:
answer = llm_text(get_llm("heavy").invoke(prompt)).strip()
print(answer)

Mitochondria are the organelles that produce most of the cell's ATP through cellular respiration. [1]


## Grounding verification (is every claim supported by the context?)

In [5]:
from app.agent import _json
v = _json(llm_text(get_llm("heavy").invoke(
    "TASK:VERIFY\nReply ONLY JSON {\"supported_ratio\":0-1,\"verdict\":\"grounded|ungrounded\"}.\n"
    f"ANSWER: {answer}\nCONTEXT: {_ctx_block(relevant)}")))
print(v)

{'supported_ratio': 1.0, 'verdict': 'grounded'}


## Fail-closed demo: a question the material cannot answer

In [6]:
q2 = "Who won the football world cup in 1998?"
h2 = HybridRetriever(where=build_where(["BIO101"]), top_k=3).search(q2)
p2 = prompt.split("CONTEXT:")[0] + f"CONTEXT:\n{_ctx_block([{'doc':d,'score':s} for d,s in h2])}\nQUESTION: {q2}\n"
print(llm_text(get_llm("heavy").invoke(p2)).strip())

INSUFFICIENT_CONTEXT
